# Karaoke-ifier Studio · Automatic Lyrics Edition
Single-page six-stem karaoke app with live microphone effects, cached MP3 playback and LRCLIB automatic synchronized lyrics.


In [ ]:
!apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null
%pip -q install "fastapi>=0.110,<1" "uvicorn>=0.27" "python-multipart>=0.0.9" "demucs==4.0.1" "soundfile>=0.12"
import torch
print("CUDA:",torch.cuda.is_available())


In [ ]:
import base64,io,zipfile
from pathlib import Path
DATA = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(DATA))) as z:
    z.extractall("/content")
print("Project updated at /content/Karaokeifier_Studio")


In [ ]:
import subprocess,sys,time,requests,os
try:
    old_server.terminate()
    old_server.wait(timeout=4)
except (NameError,Exception):
    pass
# Stop old service on port 8000 if it was launched by a previous notebook.
subprocess.run(["bash","-lc","fuser -k 8000/tcp 2>/dev/null || true"],stdout=subprocess.DEVNULL)
log=open("/content/karaoke_server.log","w")
old_server=subprocess.Popen([sys.executable,"-u","-m","uvicorn","app:app","--host","0.0.0.0","--port","8000"],cwd="/content/Karaokeifier_Studio",stdout=log,stderr=subprocess.STDOUT)
time.sleep(4)
r=requests.get("http://127.0.0.1:8000/api/health",timeout=10)
print("Status:",r.status_code,r.json())


In [ ]:
import os,subprocess,time,re,requests
if not os.path.exists("/content/cloudflared"):
    r=requests.get("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",timeout=60)
    r.raise_for_status()
    open("/content/cloudflared","wb").write(r.content)
    os.chmod("/content/cloudflared",0o755)
try:
    tunnel.terminate()
except (NameError,Exception):
    pass
tunnel_log=open("/content/karaoke_tunnel.log","w")
tunnel=subprocess.Popen(["/content/cloudflared","tunnel","--url","http://127.0.0.1:8000","--no-autoupdate"],stdout=tunnel_log,stderr=subprocess.STDOUT)
url=None
for _ in range(35):
    time.sleep(1)
    content=open("/content/karaoke_tunnel.log").read()
    m=re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",content)
    if m:
        url=m.group(0);break
print("Open Karaoke-ifier:",url or "Tunnel unavailable; inspect /content/karaoke_tunnel.log")
print("Keep this Colab runtime active while using the site.")


### Tips
- Use a 20–30-second clip to check separation first.
- First separation downloads pretrained weights; repeated uploads of identical audio use the cache until the runtime is reset.
- During playback, only selected MP3 stems are downloaded; WAV downloads remain lossless.
- Use HTTPS and headphones for live mic.
- Check `!tail -n 70 /content/karaoke_server.log` for failures.
